# System rekomendacji

- Content Based Filtering 
- Collaborative Filtering

## Content Based Filtering
- Załadowanie embeddings
- Załadowanie HSNW Index
- Zamienienie tekstu od użytkownika na embeddings
- Wybranie k najlepszych gier

### Tworzenie Embeddings
https://www.sbert.net
- Wybranie najważniejszych informacji (kolumn) i stworzenie z nich tekstu
  - Tytuł
  - Opis
  - Gatunki
  - Kategorie
  - Developer
  - Platformy
  - Dostępne treści
  - Języki
- Encoding tekstu algorytmem all-MiniLM-L6-v2
- Dostajemy embeddings i zapisujemy do pliku

In [ ]:
import json

# wczytanie danych
with open("example_games.json", "r", encoding="utf-8") as f:
    games = json.load(f)

In [ ]:
def create_embedding_text(game):
    text = f"""
Title: {game['title']}

Short description:
{game['description_short']}

Detailed description:
{game['description_long']}

Genre:
{game['main_genre']}

Categories:
{', '.join(game['categories'])}

Developer:
{game['developer']}

Platforms:
{', '.join(game['platforms'])}

Content:
{', '.join(game['content_warnings'])}

Supported languages:
{', '.join(game['supported_languages'])}
"""

    return " ".join(text.split())


# dodanie tekstu pod embedding
for game in games:
    game["embedding_text"] = create_embedding_text(game)


# zapis nowego pliku
with open("games_with_embeddings_text.json", "w", encoding="utf-8") as f:
    json.dump(games, f, ensure_ascii=False, indent=4)

In [ ]:
embeddings_text = [game["embedding_text"] for game in games]

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(embeddings_text)
np.save("embeddings.npy", embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10829.57it/s]


In [ ]:
embeddings = np.load(f"embeddings.npy")

### HNSW
https://github.com/nmslib/hnswlib

In [68]:
import hnswlib

In [69]:
index = hnswlib.Index(space='cosine', dim=embeddings.shape[1])

In [95]:
index.load_index("hnsw_index.bin", embeddings.shape[1])

ef_construction - Jakość budowy indeksu; wyższa daje lepszy indeks, ale wolniejszą budowę
M - Liczba połączeń każdego węzła; większa wartość zwiększa jakość kosztem pamięci

In [70]:
index.init_index(max_elements=len(embeddings), ef_construction=200, M=16)

In [71]:
index.add_items(embeddings)

In [72]:
index.save_index("hnsw_index.bin")

### Test

In [96]:
text = "Strzelaj z pistoletu w obiekcie black"

embedded_text = model.encode(text)

In [97]:
labels, distances = index.knn_query(embedded_text, k = 3)

In [98]:
labels

array([[4, 3, 2]], dtype=uint64)

In [99]:
distances

array([[0.5341225, 0.5469894, 0.5498463]], dtype=float32)

In [64]:
df['embedding_text'][47]

'Nazwa gry: Quake II\nGatunki: Akcja\nKategorie: Jednoosobowa, Wieloosobowa, PvP, PvP przez internet, PvP na wspólnym/dzielonym ekranie, Kooperacja, Kooperacja przez internet, Kooperacja na wspólnym/dzielonym ekranie, Wspólny/dzielony ekran, Wieloplatformowa wieloosobowa, Pełna obsługa kontrolerów, Karty kolekcjonerskie Steam, Dostosowywanie kamery, Grywalne bez działań ograniczonych czasowo, Zapis w dowolnym momencie, Dźwięk stereo, Dźwięk przestrzenny, Steam Cloud, Remote Play Together, Udostępnianie gier\nKrótki opis: Jesteś ostatnią nadzieją ludzkości na powstrzymanie Stroggów, wrogich kosmitów toczących wojnę z Ziemią. Zagraj w tę wojskową pierwszoosobową strzelankę SF przystosowaną do współczesnych platform sprzętowych i oferującą m. in. ulepszoną grafikę, nową kampanię i sieciowy tryb wieloosobowy/współpracy.\nOpis gry: Gra Quake II id Software z 1997 roku jest docenioną pierwszoosobową strzelanką z całkiem nową fabułą i oprawą SF. Teraz możesz doświadczyć autentycznej, ulepszon

In [90]:
text = "Gra strategiczna, w której trzeba podbijać terytoria świata i rozwijać swoje imperium. Trzeba mądrze zarządzać zasobami, planować ataki i obronę, a także podejmować decyzje polityczne. Gra jest sprzed 15lat i nie ma multiplayera"

embedded_text = model.encode(text)
labels, distances = index.knn_query(embedded_text, k = 5)

In [91]:
labels

array([[39, 37, 32,  0,  1]], dtype=uint64)

In [92]:
distances

array([[0.35888344, 0.37648863, 0.39053535, 0.39875925, 0.40583193]],
      dtype=float32)

In [94]:
df['embedding_text'][37]

'Nazwa gry: Space Empires V\nGatunki: Strategie\nKategorie: Jednoosobowa, Wieloosobowa, Udostępnianie gier\nKrótki opis: Space Empires V to najnowsza część serii Space Empires. Nowy epizod gry to całkowicie odmieniony interfejs i trójwymiarowe środowisko renderowane w czasie rzeczywistym. Oglądaj sceny walk, którym towarzyszą bardzo szczegółowe, realistyczne efekty. Rozwijaj, eksploruj, eksploatuj i eliminuj w ogromnej, tętniącej życiem galaktyce.\nOpis gry: Space Empires V to najnowsza część serii Space Empires. Nowy epizod gry to całkowicie odmieniony interfejs i trójwymiarowe środowisko renderowane w czasie rzeczywistym. Oglądaj sceny walk, którym towarzyszą bardzo szczegółowe, realistyczne efekty. Rozwijaj, eksploruj, eksploatuj i eliminuj w ogromnej, tętniącej życiem galaktyce. Nowości to między innymi możliwość zawierania sojuszy politycznych z wieloma imperiami, system projektowania okrętów top-down i mapa z heksagonalną siatką. Na prośbę graczy gra jest w pełni „modyfikowalna”,